In [17]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [18]:
llm = ChatGoogleGenerativeAI(model = "gemma-4-31b-it", temperature = 0)

In [19]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name:str)->str:
    """Look up a product by name and return its price, rating, stock and description"""

    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product '{name}' not found."
    return str(p)

agent = create_agent(
    llm, 
    tools = [get_product],
    system_prompt = "You are a helpful product assistant for online tech store.",
)

In [20]:
# creating a function to ask agent 
def ask_question(question:str)->str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [21]:
ask_question("What is the price of wireless headphones?")

The price of the wireless headphones is $79.99.


In [23]:
# we are creating a inmemory for the agent so it remembers the previous context of the conversation. This is useful for multi-turn conversations where the agent needs to remember what was said earlier in the conversation.

from langgraph.checkpoint.memory import InMemorySaver

# creating agent with memory
agent_with_memory = create_agent(
    llm,
    tools = [get_product],
    system_prompt = "You are a helpful product assistant for online tech store.",
    checkpointer = InMemorySaver()
)

def ask2(question:str)->str:
    # create some id for the conversation thread
    config = {"configurable":{"thread_id":"product_query_thread"}}
    result = agent_with_memory.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
    print(result["messages"][-1].content)

In [24]:
ask2("What is the price of wireless headphones?")

The price of the wireless headphones is $79.99.


In [25]:
# lets see if the agent remembers the previous context of the conversation. 
ask2("What is the description of this product?")

[{'type': 'thinking', 'thinking': 'The user is asking for the description of "this product", referring to the wireless headphones mentioned in the previous turn.\nI already have the product information from the previous `get_product` call.\nThe description is \'Over-ear Bluetooth, 30-hr battery, active noise cancellation.\''}, {'type': 'text', 'text': 'The wireless headphones are over-ear Bluetooth headphones with a 30-hour battery life and active noise cancellation.'}]
